# Planck compressed CMB 3x3 reconstruction

This section rebuilds the compressed Planck CMB data vectors and covariance matrices from the `planck_baseline` Cobaya run.

Directions computed below:

1. $(\omega_b, \omega_m, 1/\theta_s)$
2. $(\omega_b, \omega_m, \theta_s)$
3. $(\omega_b, \omega_m, 1/\theta_{\rm drag})$

Conventions:

- `omega_m` means $\Omega_m h^2$.
- `theta_s` is taken from Cobaya/Class `theta_s_100 / 100`.
- `theta_drag` is recomputed with `classy` as $r_{\rm drag}/D_M(z_{\rm drag})$, with $D_M=(1+z)D_A$; the compressed direction below is `1/theta_drag`.

In [1]:
import classy
from classy import Class
import inspect

print("classy module file:", getattr(classy, "__file__", None))
print("Class object:", Class)
print("Class module:", getattr(Class, "__module__", None))
print("Class source/file:", inspect.getfile(Class))


classy module file: /Users/williamgiare/cosmo/lib/python3.13/site-packages/classy/__init__.py
Class object: <class 'classy._classy.Class'>
Class module: classy._classy
Class source/file: /Users/williamgiare/cosmo/lib/python3.13/site-packages/classy/_classy.cpython-313-darwin.so


In [2]:
from pathlib import Path
import numpy as np

CHAIN_ROOT = Path("planck_baseline/chains/full_planck")
CHAIN_FILES = sorted(CHAIN_ROOT.parent.glob(CHAIN_ROOT.name + ".[0-9]*.txt"))
COVMAT_FILE = CHAIN_ROOT.with_suffix(".covmat")
IGNORE_ROWS = 0.3

print("Chain files:")
for f in CHAIN_FILES:
    print(" -", f)
print("Covmat:", COVMAT_FILE)


Chain files:
 - planck_baseline/chains/full_planck.1.txt
 - planck_baseline/chains/full_planck.2.txt
 - planck_baseline/chains/full_planck.3.txt
 - planck_baseline/chains/full_planck.4.txt
Covmat: planck_baseline/chains/full_planck.covmat


In [3]:
def read_cobaya_txt_chains(chain_files, ignore_rows=0.0):
    """Read Cobaya/GetDist-style weighted text chains.

    Returns names, samples, weights, minuslogpost.
    Column 0 is weight and column 1 is minuslogpost.
    """
    all_blocks = []
    names = None
    for filename in chain_files:
        with open(filename, "r") as f:
            file_names = None
            rows = []
            for line in f:
                if not line.strip():
                    continue
                if line.startswith("#"):
                    file_names = line[1:].split()
                    continue
                rows.append([float(x) for x in line.split()])
        if file_names is None:
            raise ValueError(f"No header found in {filename}")
        if names is None:
            names = file_names
        elif names != file_names:
            raise ValueError(f"Header mismatch in {filename}")

        arr = np.asarray(rows, dtype=float)
        if ignore_rows:
            arr = arr[int(np.floor(ignore_rows * len(arr))):]
        all_blocks.append(arr)

    arr = np.vstack(all_blocks)
    weights = arr[:, 0]
    minuslogpost = arr[:, 1]
    samples = arr[:, 2:]
    param_names = names[2:]
    return param_names, samples, weights, minuslogpost


def read_covmat(covmat_file):
    """Read the Cobaya proposal/sample covariance matrix file."""
    with open(covmat_file, "r") as f:
        header = None
        rows = []
        for line in f:
            if not line.strip():
                continue
            if line.startswith("#"):
                header = line[1:].split()
            else:
                rows.append([float(x) for x in line.split()])
    if header is None:
        raise ValueError(f"No header found in {covmat_file}")
    return header, np.asarray(rows, dtype=float)


def weighted_mean_cov(x, weights, ddof=0):
    """Weighted mean and covariance, using Cobaya/GetDist chain weights."""
    x = np.asarray(x, dtype=float)
    weights = np.asarray(weights, dtype=float)
    mean = np.average(x, axis=0, weights=weights)
    dx = x - mean
    cov = (dx * weights[:, None]).T @ dx / np.sum(weights)
    if ddof:
        # Kish effective sample-size correction. Leave at zero for ML convention.
        neff = np.sum(weights) ** 2 / np.sum(weights ** 2)
        cov *= neff / max(neff - ddof, 1)
    return mean, cov


def print_vector_cov(label, names, vector, cov):
    sig = np.sqrt(np.diag(cov))
    corr = cov / np.outer(sig, sig)
    print("\n" + "=" * 90)
    print(label)
    print("parameters:", names)
    print("\nData vector / weighted mean:")
    for name, value, sigma in zip(names, vector, sig):
        print(f"  {name:16s} {value:.12e}    sigma={sigma:.12e}")
    print("\nCovariance matrix:")
    print(np.array2string(cov, precision=12, suppress_small=False, max_line_width=140))
    print("\nCorrelation matrix:")
    print(np.array2string(corr, precision=6, suppress_small=False, max_line_width=140))


In [4]:
try:
    from getdist import loadMCSamples
    gd_sample = loadMCSamples(str(CHAIN_ROOT), settings={"ignore_rows": IGNORE_ROWS})
    gd_names = gd_sample.getParamNames().list()
    gd_weights = gd_sample.weights
    gd_samples = gd_sample.samples
    print("Loaded with GetDist")
    names = gd_names
    samples = gd_samples
    weights = gd_weights
except Exception as exc:
    print("GetDist unavailable or failed; using NumPy fallback:", repr(exc))
    names, samples, weights, minuslogpost = read_cobaya_txt_chains(CHAIN_FILES, ignore_rows=IGNORE_ROWS)

name_to_i = {name: i for i, name in enumerate(names)}
print(f"Loaded {samples.shape[0]} rows after ignore_rows={IGNORE_ROWS}; sum(weights)={np.sum(weights):.0f}")
print("Available derived geometry columns:", [p for p in ["theta_s_100", "theta_star_100", "rs_drag", "z_d", "rs_star", "da_star", "z_star"] if p in name_to_i])

covmat_names, run_covmat = read_covmat(COVMAT_FILE)
print(f"Read .covmat with shape {run_covmat.shape} and {len(covmat_names)} parameters")


Loaded with GetDist
Loaded 21804 rows after ignore_rows=0.3; sum(weights)=61624
Available derived geometry columns: ['theta_s_100', 'theta_star_100', 'rs_drag', 'z_d', 'rs_star', 'da_star', 'z_star']
Read .covmat with shape (27, 27) and 27 parameters


In [5]:
def col(name):
    if name not in name_to_i:
        raise KeyError(f"Missing parameter '{name}' in chains")
    return samples[:, name_to_i[name]]

omega_b = col("omega_b")
omega_m = col("omegamh2") if "omegamh2" in name_to_i else col("Omega_m") * (col("H0") / 100.0) ** 2
theta_s = col("theta_s_100") / 100.0
inv_theta_s = 1.0 / theta_s

# Quick consistency check against the run .covmat for the sampled base parameters.
for subset in [("omega_b", "omega_cdm", "theta_s_100"), ("omega_b", "omega_cdm")]:
    if all(p in covmat_names for p in subset):
        idx = [covmat_names.index(p) for p in subset]
        print("\n.covmat sub-block for", subset)
        print(np.array2string(run_covmat[np.ix_(idx, idx)], precision=12, suppress_small=False))



.covmat sub-block for ('omega_b', 'omega_cdm', 'theta_s_100')
[[ 2.302838186109e-08 -1.252940452190e-07  1.155946081360e-08]
 [-1.252940452190e-07  1.930913194237e-06 -1.179777074120e-07]
 [ 1.155946081360e-08 -1.179777074120e-07  9.213089899613e-08]]

.covmat sub-block for ('omega_b', 'omega_cdm')
[[ 2.302838186109e-08 -1.252940452190e-07]
 [-1.252940452190e-07  1.930913194237e-06]]


In [6]:
# Recompute theta_drag with CLASS/classy for every chain sample.
# We pass the sampled background parameters to CLASS, evaluate D_A(z_d), and use
# D_M(z_d) = (1 + z_d) D_A(z_d). The compressed direction uses 1/theta_drag.

CLASS_EXTRA_ARGS = {
    "N_ncdm": 1,
    "N_ur": 2.0328,
    "m_ncdm": 0.06,
}


def class_params_for_row(i):
    params = {
        "H0": float(col("H0")[i]),
        "omega_b": float(col("omega_b")[i]),
        "omega_cdm": float(col("omega_cdm")[i]),
        "Omega_k": 0.0,
        **CLASS_EXTRA_ARGS,
    }
    # These are not needed for distances, but including them keeps the setup close
    # to the Cobaya run and lets CLASS initialize consistently if thermodynamics
    # quantities are requested internally.
    if "n_s" in name_to_i:
        params["n_s"] = float(col("n_s")[i])
    if "A_s" in name_to_i:
        params["A_s"] = float(col("A_s")[i])
    elif "logA" in name_to_i:
        params["A_s"] = 1e-10 * np.exp(float(col("logA")[i]))
    if "tau_reio" in name_to_i:
        params["tau_reio"] = float(col("tau_reio")[i])
    return params


def theta_drag_from_classy(i):
    cosmo = Class()
    try:
        cosmo.set(class_params_for_row(i))
        cosmo.compute()
        z = float(col("z_d")[i])
        r_drag = float(col("rs_drag")[i])
        D_A = cosmo.angular_distance(z)
        D_M = (1.0 + z) * D_A
        return r_drag / D_M
    finally:
        # CLASS allocates C-side memory; always clean it between samples.
        try:
            cosmo.struct_cleanup()
            cosmo.empty()
        except Exception:
            pass


n = samples.shape[0]
theta_drag = np.empty(n, dtype=float)
for i in range(n):
    theta_drag[i] = theta_drag_from_classy(i)
    if (i + 1) % 1000 == 0 or i + 1 == n:
        print(f"CLASS theta_drag: {i + 1}/{n}")

inv_theta_drag = 1.0 / theta_drag

print("theta_drag recomputed with CLASS/classy from chain parameters")
print(f"  mean(theta_drag) = {np.average(theta_drag, weights=weights):.12e}")
print(f"  mean(1/theta_drag) = {np.average(inv_theta_drag, weights=weights):.12e}")


CLASS theta_drag: 1000/21804
CLASS theta_drag: 2000/21804
CLASS theta_drag: 3000/21804
CLASS theta_drag: 4000/21804
CLASS theta_drag: 5000/21804
CLASS theta_drag: 6000/21804
CLASS theta_drag: 7000/21804
CLASS theta_drag: 8000/21804
CLASS theta_drag: 9000/21804
CLASS theta_drag: 10000/21804
CLASS theta_drag: 11000/21804
CLASS theta_drag: 12000/21804
CLASS theta_drag: 13000/21804
CLASS theta_drag: 14000/21804
CLASS theta_drag: 15000/21804
CLASS theta_drag: 16000/21804
CLASS theta_drag: 17000/21804
CLASS theta_drag: 18000/21804
CLASS theta_drag: 19000/21804
CLASS theta_drag: 20000/21804
CLASS theta_drag: 21000/21804
CLASS theta_drag: 21804/21804
theta_drag recomputed with CLASS/classy from chain parameters
  mean(theta_drag) = 1.060831312272e-02
  mean(1/theta_drag) = 9.426570752433e+01


In [7]:
directions = {
    "1) omega_b, omega_m, 1/theta_s": (
        ["omega_b", "omega_m", "1/theta_s"],
        np.column_stack([omega_b, omega_m, inv_theta_s]),
    ),
    "2) omega_b, omega_m, theta_s": (
        ["omega_b", "omega_m", "theta_s"],
        np.column_stack([omega_b, omega_m, theta_s]),
    ),
    "3) omega_b, omega_m, 1/theta_drag": (
        ["omega_b", "omega_m", "1/theta_drag"],
        np.column_stack([omega_b, omega_m, inv_theta_drag]),
    ),
}

compressed_results = {}
for label, (param_names, values) in directions.items():
    mean, cov = weighted_mean_cov(values, weights)
    compressed_results[label] = {"params": param_names, "mean": mean, "cov": cov, "inv_cov": np.linalg.inv(cov)}
    print_vector_cov(label, param_names, mean, cov)



1) omega_b, omega_m, 1/theta_s
parameters: ['omega_b', 'omega_m', '1/theta_s']

Data vector / weighted mean:
  omega_b          2.234245077095e-02    sigma=1.518422587775e-04
  omega_m          1.431188525240e-01    sigma=1.298390284788e-03
  1/theta_s        9.598406806806e+01    sigma=2.788440800133e-02

Covariance matrix:
[[ 2.305607155067e-08 -1.018951554651e-07 -1.040680516510e-06]
 [-1.018951554651e-07  1.685817331632e-06  9.610322661471e-06]
 [-1.040680516510e-06  9.610322661471e-06  7.775402095849e-04]]

Correlation matrix:
[[ 1.       -0.516839 -0.24579 ]
 [-0.516839  1.        0.265443]
 [-0.24579   0.265443  1.      ]]

2) omega_b, omega_m, theta_s
parameters: ['omega_b', 'omega_m', 'theta_s']

Data vector / weighted mean:
  omega_b          2.234245077095e-02    sigma=1.518422587775e-04
  omega_m          1.431188525240e-01    sigma=1.298390284788e-03
  theta_s          1.041839655814e-02    sigma=3.026667938135e-06

Covariance matrix:
[[ 2.305607155067e-08 -1.018951554651

/var/folders/9v/2pq2bh6s36l34yfxzqczyg9m0000gn/T/ipykernel_62452/1014942676.py:63: RuntimeWarning: divide by zero encountered in matmul
  cov = (dx * weights[:, None]).T @ dx / np.sum(weights)
/var/folders/9v/2pq2bh6s36l34yfxzqczyg9m0000gn/T/ipykernel_62452/1014942676.py:63: RuntimeWarning: overflow encountered in matmul
  cov = (dx * weights[:, None]).T @ dx / np.sum(weights)
/var/folders/9v/2pq2bh6s36l34yfxzqczyg9m0000gn/T/ipykernel_62452/1014942676.py:63: RuntimeWarning: invalid value encountered in matmul
  cov = (dx * weights[:, None]).T @ dx / np.sum(weights)


In [12]:
for label, result in compressed_results.items():
    print("\n" + "#" * 90)
    print("#", label)
    print("params =", result["params"])
    print("data = np.array(")
    print(np.array2string(result["mean"], precision=12, separator=", ", max_line_width=120))
    print(", dtype=float)")
    print("cov = np.array(")
    print(np.array2string(result["cov"], precision=12, separator=", ", max_line_width=120))
    print(", dtype=float)")


##########################################################################################
# 1) omega_b, omega_m, 1/theta_s
params = ['omega_b', 'omega_m', '1/theta_s']
data = np.array(
[2.234245077095e-02, 1.431188525240e-01, 9.598406806806e+01]
, dtype=float)
cov = np.array(
[[ 2.305607155067e-08, -1.018951554651e-07, -1.040680516510e-06],
 [-1.018951554651e-07,  1.685817331632e-06,  9.610322661471e-06],
 [-1.040680516510e-06,  9.610322661471e-06,  7.775402095849e-04]]
, dtype=float)

##########################################################################################
# 2) omega_b, omega_m, theta_s
params = ['omega_b', 'omega_m', 'theta_s']
data = np.array(
[0.022342450771, 0.143118852524, 0.010418396558]
, dtype=float)
cov = np.array(
[[ 2.305607155067e-08, -1.018951554651e-07,  1.129581718825e-10],
 [-1.018951554651e-07,  1.685817331632e-06, -1.043110754836e-09],
 [ 1.129581718825e-10, -1.043110754836e-09,  9.160718807732e-12]]
, dtype=float)

###############################

In [11]:
outdir = Path("compressed_data_vectors")
outdir.mkdir(exist_ok=True)

save_map = {
    "2) omega_b, omega_m, theta_s": "wb_wm_Thetas.dat",
    "1) omega_b, omega_m, 1/theta_s": "wb_wm_invThetas.dat",
    "3) omega_b, omega_m, 1/theta_drag": "wb_wm_invThetadrag.dat",
}

for key, filename in save_map.items():
    result = compressed_results[key]
    path = outdir / filename

    with open(path, "w") as f:
        f.write("# compressed CMB 3x3 data vector\n")
        f.write("# params: " + " ".join(result["params"]) + "\n")
        f.write("# data\n")
        np.savetxt(f, result["mean"][None, :], fmt="%.16e")
        f.write("# cov\n")
        np.savetxt(f, result["cov"], fmt="%.16e")

    print(f"saved {path}")


saved compressed_data_vectors/wb_wm_Thetas.dat
saved compressed_data_vectors/wb_wm_invThetas.dat
saved compressed_data_vectors/wb_wm_invThetadrag.dat
